# Ground Truth Pipeline — Rename + Merge + Component Heights

### What this notebook does
1. **Cell 1** — Renames your masks to consecutive `pole_001_mask.png` … `pole_037_mask.png`
2. **Cell 2** — Merges LiDAR masks + GSV masks into one fused ground truth per pole
3. **Cell 3** — Extracts component heights above ground from the LAS + fused masks
4. **Cell 4** — Saves `ground_truth.csv` (one row per component per pole) — ready for ML

### Ground truth fusion rule
| Situation | What wins |
|---|---|
| LiDAR mask has label, GSV mask empty | LiDAR wins |
| GSV mask has label, LiDAR mask empty | GSV wins |
| Both have same label | Either (consistent) |
| Both have conflicting labels | **LiDAR wins** (3D source is more precise) |


## Cell 1 — Rename masks to consecutive numbering

In [ ]:
import os, shutil, cv2
import numpy as np

LIDAR_MASK_DIR = 'pole_masks'        # your labelled masks from further.ipynb
GSV_MASK_DIR   = 'gsv_masks'         # your Google mapper masks (same structure)
OUT_LIDAR_DIR  = 'gt_lidar_masks'    # renamed LiDAR masks
OUT_GSV_DIR    = 'gt_gsv_masks'      # renamed GSV masks

os.makedirs(OUT_LIDAR_DIR, exist_ok=True)
os.makedirs(OUT_GSV_DIR,   exist_ok=True)

# ── Mapping: mask file number → consecutive new ID ────────────────────────────
# further.ipynb numbered masks by CSV row index (0-based, skipping tree rows)
# Duplicates (_1 suffix) = pole was re-labelled → _1 is the newer version, use it

MASK_NUM_TO_NEW_ID = {
     0:  1,  1:  2,  3:  3,  4:  4,  5:  5,  6:  6,
     9:  7, 10:  8, 11:  9, 12: 10, 13: 11, 14: 12,
    15: 13, 16: 14, 17: 15, 19: 16, 22: 17, 23: 18,
    24: 19, 25: 20, 27: 21, 28: 22, 29: 23, 30: 24,
    31: 25, 32: 26, 33: 27, 36: 28, 37: 29, 38: 30,
    40: 31, 41: 32, 42: 33, 43: 34, 44: 35, 45: 36,
    46: 37,
}

# Poles that were re-labelled (pole_X_1_mask.png is the newer version)
REDONE = {9, 14, 16, 17, 18, 28}  # new IDs that had _1 re-done

def rename_masks(src_dir, dst_dir, label='LiDAR'):
    """Rename mask files from further.ipynb numbering to consecutive 1-37."""
    if not os.path.exists(src_dir):
        print(f'  {src_dir} not found — skipping')
        return {}

    renamed = {}
    for mask_num, new_id in sorted(MASK_NUM_TO_NEW_ID.items()):
        # If this pole was re-done, prefer the _1 version
        if new_id in REDONE:
            candidates = [
                f'pole_{mask_num}_1_mask.png',  # re-done (preferred)
                f'pole_{mask_num}_mask.png',    # original fallback
            ]
        else:
            candidates = [f'pole_{mask_num}_mask.png']

        src_file = None
        for cand in candidates:
            path = os.path.join(src_dir, cand)
            if os.path.exists(path):
                src_file = path
                break

        dst_file = os.path.join(dst_dir, f'pole_{new_id:03d}_mask.png')

        if src_file is None:
            print(f'  [{label}] Pole {new_id:02d}: NOT FOUND (mask_num={mask_num})')
            continue

        shutil.copy2(src_file, dst_file)
        renamed[new_id] = dst_file

        # Quick check: is the mask non-empty?
        m = cv2.imread(dst_file, cv2.IMREAD_GRAYSCALE)
        if m is None:
            print(f'  [{label}] Pole {new_id:02d}: COPIED but unreadable!')
        else:
            n_labelled = (m > 0).sum()
            print(f'  [{label}] Pole {new_id:02d}: {n_labelled:>6} labelled pixels  ← {os.path.basename(src_file)}')

    return renamed

print('=== Renaming LiDAR masks ===')
lidar_renamed = rename_masks(LIDAR_MASK_DIR, OUT_LIDAR_DIR, 'LiDAR')

print()
print('=== Renaming GSV masks ===')
gsv_renamed = rename_masks(GSV_MASK_DIR, OUT_GSV_DIR, 'GSV')

print(f'\nLiDAR masks renamed: {len(lidar_renamed)}/37')
print(f'GSV   masks renamed: {len(gsv_renamed)}/37')
print('Run Cell 2 to fuse both sources into one ground truth.')


## Cell 2 — Fuse LiDAR + GSV masks into one ground truth mask per pole

In [ ]:
FUSED_DIR = 'gt_fused_masks'
os.makedirs(FUSED_DIR, exist_ok=True)

# LiDAR wins on conflict (3D source more geometrically accurate)
# GSV fills in where LiDAR mask is empty (e.g. transformer, crossarm detail)

# Label IDs — must match what BOTH labelers used
# LiDAR labeler (further.ipynb):  1=SHAFT 2=CROSSARM 3=PRIMARY 4=SECONDARY
#                                  5=NEUTRAL 6=DOWNGUY 7=ANCHOR 8=TRANSFORMER
# GSV labeler (further.ipynb):    same taxonomy assumed
LABEL_NAMES = {
    1:'SHAFT', 2:'CROSSARM', 3:'PRIMARY', 4:'SECONDARY',
    5:'NEUTRAL', 6:'DOWNGUY', 7:'ANCHOR', 8:'TRANSFORMER'
}

fusion_report = []

print(f'Fusing masks for poles 1–37 → {FUSED_DIR}/')
for new_id in range(1, 38):
    lidar_path = os.path.join(OUT_LIDAR_DIR, f'pole_{new_id:03d}_mask.png')
    gsv_path   = os.path.join(OUT_GSV_DIR,   f'pole_{new_id:03d}_mask.png')

    lidar_mask = cv2.imread(lidar_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(lidar_path) else None
    gsv_mask   = cv2.imread(gsv_path,   cv2.IMREAD_GRAYSCALE) if os.path.exists(gsv_path)   else None

    if lidar_mask is None and gsv_mask is None:
        print(f'  Pole {new_id:02d}: no mask from either source — skipped')
        fusion_report.append({'pole_id': new_id, 'source': 'NONE',
                               'lidar_px': 0, 'gsv_px': 0, 'fused_px': 0})
        continue

    # Use whichever is available as base; if both exist, fuse
    if lidar_mask is None:
        fused = gsv_mask.copy(); src = 'GSV_ONLY'
    elif gsv_mask is None:
        fused = lidar_mask.copy(); src = 'LIDAR_ONLY'
    else:
        # Both available: start with GSV, overwrite with LiDAR where LiDAR != 0
        fused = gsv_mask.copy()
        lidar_nz = lidar_mask > 0
        fused[lidar_nz] = lidar_mask[lidar_nz]  # LiDAR wins
        src = 'FUSED'

    dst_path = os.path.join(FUSED_DIR, f'pole_{new_id:03d}_mask.png')
    cv2.imwrite(dst_path, fused)

    n_lidar = int((lidar_mask > 0).sum()) if lidar_mask is not None else 0
    n_gsv   = int((gsv_mask   > 0).sum()) if gsv_mask   is not None else 0
    n_fused = int((fused > 0).sum())

    # Per-class breakdown
    classes_present = [LABEL_NAMES[v] for v in np.unique(fused) if v > 0 and v in LABEL_NAMES]

    print(f'  Pole {new_id:02d} [{src:>11s}]: LiDAR={n_lidar:>5} GSV={n_gsv:>5} '
          f'Fused={n_fused:>5}  [{", ".join(classes_present)}]')

    fusion_report.append({'pole_id': new_id, 'source': src,
                          'lidar_px': n_lidar, 'gsv_px': n_gsv, 'fused_px': n_fused,
                          'components': ', '.join(classes_present)})

import pandas as pd
pd.DataFrame(fusion_report).to_csv('fusion_report.csv', index=False)
print('\nfusion_report.csv saved. Run Cell 3 to extract heights.')


## Cell 3 — Extract component heights above ground from LAS + fused masks

In [ ]:
import laspy
from sklearn.decomposition import PCA

IN_LAS = 'allwires_v1.las'

# Render geometry constants — MUST match what Cell 2 of the labeller used
# (from full_component_labeller_v2.ipynb)
FIXED_U_M  = 80.0
FIXED_Z_M  = 22.0
IMG_W      = 800
IMG_H      = 220
SCALE_MPX  = FIXED_U_M / IMG_W   # 0.1 m/px

print(f'Loading {IN_LAS} ...')
las      = laspy.read(IN_LAS)
pts      = np.vstack([las.x, las.y, las.z]).T
cls_las  = np.array(las.classification, dtype=np.uint8)
uid      = np.array(las.user_data,      dtype=np.int32)

# Original IDs present in LAS → consecutive new ID mapping
orig_ids = [1,2,4,5,6,7,10,11,12,13,14,15,16,17,18,20,23,24,25,26,28,29,
            30,31,32,33,34,37,38,39,41,42,43,44,45,46,47]
ORIG_TO_NEW = {orig: new for new, orig in enumerate(orig_ids, start=1)}
NEW_TO_ORIG = {v: k for k, v in ORIG_TO_NEW.items()}

# Global road PCA (same as labeller used)
pole_mask  = cls_las == 65
all_pole_xy = np.array([pts[pole_mask & (uid == oid), :2].mean(axis=0)
                         for oid in orig_ids])
road_pca    = PCA(n_components=2).fit(all_pole_xy)
road_u_vec  = road_pca.components_[0]

# Full 2D tree
from scipy.spatial import cKDTree
full_tree = cKDTree(pts[:, :2])

LABEL_NAMES = {
    1:'SHAFT', 2:'CROSSARM', 3:'PRIMARY', 4:'SECONDARY',
    5:'NEUTRAL', 6:'DOWNGUY', 7:'ANCHOR', 8:'TRANSFORMER'
}

rows = []

print(f'\nExtracting component heights for {len(orig_ids)} poles...')
for new_id in range(1, 38):
    orig_pid = NEW_TO_ORIG[new_id]
    m_pole   = pole_mask & (uid == orig_pid)
    if not m_pole.any():
        continue

    p_pole   = pts[m_pole]
    pole_xy  = p_pole[:, :2].mean(axis=0)
    bot_z    = float(p_pole[:, 2].min())    # ground reference = pole base
    top_z    = float(p_pole[:, 2].max())
    pole_h   = top_z - bot_z

    # Z window for this pole (same as labeller)
    z_lo = bot_z - 1.0
    z_hi = z_lo + FIXED_Z_M

    # Pull nearby points
    raw_local = np.array(full_tree.query_ball_point(pole_xy, r=FIXED_U_M/2 + 3), dtype=np.int64)
    near_pts  = pts[raw_local]
    zm = (near_pts[:,2] >= z_lo) & (near_pts[:,2] <= z_hi)
    near_pts  = near_pts[zm]
    near_gidx = raw_local[zm]
    if len(near_pts) == 0:
        continue

    # Project same as labeller
    rel_xy = near_pts[:, :2] - pole_xy
    u_phys = rel_xy @ road_u_vec
    z_phys = near_pts[:, 2]
    u_ctr  = IMG_W // 2

    ux = np.clip((u_ctr + u_phys / SCALE_MPX).astype(int), 0, IMG_W - 1)
    vx = np.clip(((IMG_H-1) - (z_phys - z_lo) / SCALE_MPX).astype(int), 0, IMG_H - 1)

    # Load fused mask
    mask_path = os.path.join(FUSED_DIR, f'pole_{new_id:03d}_mask.png')
    if not os.path.exists(mask_path):
        continue
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        continue

    # Read label per near point from mask
    painted = mask[vx, ux]   # label id at each point position

    # Pole base row (always present)
    rows.append({
        'pole_id':          new_id,
        'orig_pole_id':     orig_pid,
        'component':        'POLE_BASE',
        'height_above_ground_m': 0.0,
        'abs_z':            round(bot_z, 3),
        'pole_height_m':    round(pole_h, 2),
        'n_points':         int(m_pole.sum()),
    })

    # Per-label extraction
    for label_id, label_name in LABEL_NAMES.items():
        label_mask = painted == label_id
        if not label_mask.any():
            continue

        label_z = z_phys[label_mask]

        rows.append({
            'pole_id':               new_id,
            'orig_pole_id':          orig_pid,
            'component':             label_name,
            # Height of the component's MEDIAN Z above pole base
            'height_above_ground_m': round(float(np.median(label_z)) - bot_z, 2),
            'abs_z':                 round(float(np.median(label_z)), 3),
            'pole_height_m':         round(pole_h, 2),
            'n_points':              int(label_mask.sum()),
        })

    labelled_comps = [LABEL_NAMES[v] for v in np.unique(painted) if v in LABEL_NAMES]
    print(f'  Pole {new_id:02d} (orig {orig_pid:2d}): base={bot_z:.1f}m  '
          f'h={pole_h:.1f}m  components=[{", ".join(labelled_comps)}]')

df = pd.DataFrame(rows)
df.to_csv('ground_truth.csv', index=False)
print(f'\nground_truth.csv saved — {len(df)} rows across {df["pole_id"].nunique()} poles')
print('\nSample:')
print(df.head(20).to_string(index=False))


## Cell 4 — Component height summary table

In [ ]:
# Pivot: one row per pole, columns = component median heights above ground

gt = pd.read_csv('ground_truth.csv')

comp_cols = ['SHAFT','CROSSARM','TRANSFORMER','PRIMARY','SECONDARY','NEUTRAL','DOWNGUY','ANCHOR']
present   = [c for c in comp_cols if c in gt['component'].values]

pivot = gt.pivot_table(
    index=['pole_id','orig_pole_id','pole_height_m'],
    columns='component',
    values='height_above_ground_m',
    aggfunc='median'
).reset_index()

pivot.columns.name = None
pivot = pivot[['pole_id','orig_pole_id','pole_height_m']
              + [c for c in comp_cols if c in pivot.columns]]

pivot.to_csv('component_heights_summary.csv', index=False)
print('component_heights_summary.csv saved\n')
print(pivot.to_string(index=False))

# Quick stats
print('\n=== Component height statistics (median above ground) ===')
for comp in present:
    sub = gt[gt['component'] == comp]['height_above_ground_m']
    if len(sub) == 0: continue
    print(f'  {comp:<14s}: n={len(sub):>2}  '
          f'min={sub.min():.1f}m  median={sub.median():.1f}m  max={sub.max():.1f}m')
